In [1]:
import all_tickers as at

gtd = at.GetTickersData(ticker='icici bank')
df = gtd.get_df_by_ticker()

In [1]:
from datetime import datetime
import yfinance as yf
import matplotlib.pyplot as plt
  
# initialize parameters
start_date = datetime(2023,8, 1)
end_date = datetime(2023, 10, 21)
  
# get the data
icici_data = yf.download('WIPRO.NS', start = start_date,
                   end = end_date)

[*********************100%%**********************]  1 of 1 completed


In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn import model_selection
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout

# Load your dataset (replace 'your_data.csv' with the actual file path)
#data = pd.read_csv('your_data.csv')

data = icici_data.copy()
# data['Date'] = pd.to_datetime(data['Date'])
# data.set_index('Date', inplace=True)



# Function to compute Relative Strength Index (RSI)
def compute_rsi(close_prices, window=5):
    # Compute price changes
    delta = close_prices.diff()

    # Calculate gains and losses
    gains = delta.where(delta > 0, 0)
    losses = -delta.where(delta < 0, 0)

    # Calculate average gains and losses
    avg_gains = gains.rolling(window=window).mean()
    avg_losses = losses.rolling(window=window).mean()

    # Calculate RSI
    rs = avg_gains / (avg_losses + 1e-10)
    rsi = 100 - (100 / (1 + rs))
    
    return rsi

# Function to compute Moving Average Convergence Divergence (MACD)
def compute_macd(close_prices, short_window=12, long_window=26, signal_window=9):
    short_ema = close_prices.ewm(span=short_window, min_periods=1, adjust=False).mean()
    long_ema = close_prices.ewm(span=long_window, min_periods=1, adjust=False).mean()
    macd = short_ema - long_ema
    signal_line = macd.ewm(span=signal_window, min_periods=1, adjust=False).mean()
    return macd - signal_line

def dependent_variable_window(data,window_size):

    dependent_variable = list()
    for i in range(len(data) - window_size + 1):

        window = data[i:i + window_size]
        dependent_variable.append(window)
    return np.array(dependent_variable)


def sliding_window_scaling(data, window_size):
    scaled_data = []

    for i in range(len(data) - window_size + 1):
        window = data[i:i + window_size]
        print(window)
        min_val = window.min()
        max_val = window.max()
        scaled_window = (window - min_val) / (max_val - min_val)
        scaled_data.append(scaled_window)

    return np.array(scaled_data)

# Create new features (e.g., moving averages)
data['SMA_50'] = data['Close'].rolling(window=50).mean()
data['SMA_200'] = data['Close'].rolling(window=200).mean()
data['RSI'] = compute_rsi(data['Close'])
data['MACD'] = compute_macd(data['Close'])

# Define the prediction horizon (7 to 14 days)
prediction_horizon = 5

# Create lagged features for the target variable (Close price)
for i in range(1, prediction_horizon + 1):
    data[f'Close_Lag_{i}'] = data['Close'].shift(-i)

# Drop rows with missing values (due to lagged features)
data.dropna(inplace=True)



# Define input features and target variable
input_features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_50', 'SMA_200', 'RSI', 'MACD']
target_variable = ['Close_Lag_1']  # Predicting the next day's closing price

# Split the data into train and test sets
X = data[input_features].values
y = data[target_variable].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Sliding window scaling for X_train and X_test
window_size = 5  # Adjust the window size as needed
X_train_scaled = sliding_window_scaling(X_train, window_size)
X_test_scaled = sliding_window_scaling(X_test, window_size)
y_train_scaled = dependent_variable_window(y_train,window_size)
y_test_scaled = dependent_variable_window(y_test,window_size)


# Reshape the data to fit the LSTM model
X_train_scaled = X_train_scaled.reshape(-1, window_size, len(input_features))
X_test_scaled = X_test_scaled.reshape(-1, window_size, len(input_features))

# X_train_scaled = pd.Series(X_train_scaled).values()
# X_test_scaled = pd.Series(X_test_scaled).values()

# Define the LSTM model
# model = keras.Sequential([
#     layers.LSTM(50, activation='relu', input_shape=(window_size, len(input_features))),
#     layers.Dense(1)  # Output layer with 1 neuron for regression
# ])

model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(5, 9)))
model.add(LSTM(50, return_sequences=True))
model.add(LSTM(50))
model.add(Dense(1))
model.compile(loss='mean_squared_error', optimizer='adam')

# Train the model
model.fit(X_train_scaled, y_train_scaled, epochs=50, batch_size=10)

# Evaluate the model on the test data
loss = model.evaluate(X_test_scaled, y_test_scaled)
print(f"Test Loss: {loss:.4f}")

# Make predictions for the next 7-14 days
last_window = X_test_scaled[-1]  # Use the last window of data in X_test_scaled
predicted_prices = []

for i in range(prediction_horizon):
    next_price = model.predict(np.expand_dims(last_window, axis=0))
    predicted_prices.append(next_price[0, 0])
    # Update the last window with the predicted price
    last_window = np.roll(last_window, shift=-1, axis=1)
    last_window[0, -1] = next_price[0, 0]

# Print the predicted prices for the next 7-14 days
print("Predicted Prices for the Next 7-14 Days:")
for i, price in enumerate(predicted_prices):
    print(f"Day {i+1}: {price}")

KeyboardInterrupt: 

In [ ]:
data

,Open,High,Low,Close,Adj Close,Volume,SMA_50,SMA_200,RSI,MACD,Close_Lag_1,Close_Lag_2,Close_Lag_3,Close_Lag_4,Close_Lag_5
Date,,,,,,,,,,,,,,,


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense
from sklearn.model_selection import GridSearchCV
from datetime import datetime
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
  
# initialize parameters
start_date = datetime(2023,4, 1)
end_date = datetime(2023, 10, 27)
  
# get the data
icici_data = yf.download('ICICIBANK.NS', start = start_date,)

# Load your dataset (e.g., from a CSV file)


# Load your dataset (e.g., from a CSV file)
data = icici_data.copy()
data = data['Close'].values  # Assuming you're interested in predicting closing prices

# Normalize the data
scaler = MinMaxScaler(feature_range=(0, 1))
data = scaler.fit_transform(data.reshape(-1, 1))

# Split the data into training and test sets
train_size = int(len(data) * 0.8)
train_data, test_data = data[:train_size], data[train_size:]

# Create sequences and labels
def create_sequences(data, seq_length):
    sequences, labels = [], []
    for i in range(len(data) - seq_length):
        seq = data[i:i + seq_length]
        label = data[i + seq_length]
        sequences.append(seq)
        labels.append(label)
    return np.array(sequences), np.array(labels)

seq_length = 5
X_train, y_train = create_sequences(train_data, seq_length)
X_test, y_test = create_sequences(test_data, seq_length)

# Build the LSTM model
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(seq_length, 1)))
model.add(LSTM(50, return_sequences=True))
model.add(LSTM(50))
model.add(Dense(1))
model.compile(loss='mean_squared_error', optimizer='adam')

# Train the model
history=model.fit(X_train, y_train, epochs=100, batch_size=64,validation_data=(X_test,y_test))
vl_loss = sum(history.history['val_loss'])
# Make predictions on the test set
test_predict = model.predict(X_test)

# Inverse transform the predictions to the original scale
test_predict = scaler.inverse_transform(test_predict)

# Plot the results
# plt.figure(figsize=(12, 6))
# plt.plot(data, label='Original Data', color='b')
# plt.plot(test_predict, label='Testing Prediction', color='r')
# plt.legend()
# plt.show()

# Predict stock prices for the next 7 days
last_sequence = X_test[-1]  # Get the last sequence from the test set
forecast_days = 5
forecast = []

for _ in range(forecast_days):
    # Predict the next day's price
    next_day_prediction = model.predict(last_sequence.reshape(1, seq_length, 1))
    
    # Inverse transform the prediction to the original scale
    next_day_prediction = scaler.inverse_transform(next_day_prediction)
    
    # Append the prediction to the forecast
    forecast.append(next_day_prediction[0][0])
    
    # Update the last sequence with the new prediction
    last_sequence = np.roll(last_sequence, shift=-1)
    last_sequence[-1] = next_day_prediction

# Print the forecasts for the next 7 days
print("Forecast for the next 7 days:")
for i, price in enumerate(forecast):
    print(f"Day {i+1}: Predicted Price = {price-vl_loss:.2f}")



[*********************100%%**********************]  1 of 1 completed
Epoch 1/100
2/2 [==============================] - 5s 925ms/step - loss: 0.4085 - val_loss: 0.2622
Epoch 2/100
2/2 [==============================] - 0s 36ms/step - loss: 0.3564 - val_loss: 0.2190
Epoch 3/100
2/2 [==============================] - 0s 38ms/step - loss: 0.3059 - val_loss: 0.1744
Epoch 4/100
2/2 [==============================] - 0s 41ms/step - loss: 0.2514 - val_loss: 0.1277
Epoch 5/100
2/2 [==============================] - 0s 32ms/step - loss: 0.1933 - val_loss: 0.0801
Epoch 6/100
2/2 [==============================] - 0s 39ms/step - loss: 0.1289 - val_loss: 0.0370
Epoch 7/100
2/2 [==============================] - 0s 38ms/step - loss: 0.0695 - val_loss: 0.0102
Epoch 8/100
2/2 [==============================] - 0s 35ms/step - loss: 0.0255 - val_loss: 0.0210
Epoch 9/100
2/2 [==============================] - 0s 35ms/step - loss: 0.0149 - val_loss: 0.0757
Epoch 10/100
2/2 [==============================

In [2]:
vl_loss = sum(history.history['val_loss'])/len(history.history['val_loss'])

In [4]:
sum(vl_loss)

TypeError: 'float' object is not iterable

In [5]:
sum(history.history['val_loss'])

12.242117907851934

In [1]:
# Importing dependencies

import numpy as np
np.random.seed(1)
# from tensorflow import set_random_seed
# set_random_seed(2)
import pandas as pd
import matplotlib.pyplot as plt
from keras.models import Sequential, load_model
from keras.layers import Dense, LSTM
from keras import optimizers
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from math import sqrt
import datetime as dt
import time
plt.style.use('ggplot')

In [2]:
from datetime import datetime
import yfinance as yf
import matplotlib.pyplot as plt

# initialize parameters
start_date = datetime(2000, 1, 1)
end_date = datetime(2023, 12, 1)

# get the data
df = yf.download('ICICIBANK.NS', start = start_date,
                   end = end_date)

[*********************100%%**********************]  1 of 1 completed


In [4]:
# Setting up an early stop
earlystop = EarlyStopping(monitor='val_loss', min_delta=0.0001, patience=80,  verbose=1, mode='min')
callbacks_list = [earlystop]

In [5]:
#Build and train the model
def fit_model(train,val,timesteps,hl,lr,batch,epochs):
    X_train = []
    Y_train = []
    X_val = []
    Y_val = []

    # Loop for training data
    for i in range(timesteps,train.shape[0]):
        X_train.append(train[i-timesteps:i])
        Y_train.append(train[i][0])
    X_train,Y_train = np.array(X_train),np.array(Y_train)

    # Loop for val data
    for i in range(timesteps,val.shape[0]):
        X_val.append(val[i-timesteps:i])
        Y_val.append(val[i][0])
    X_val,Y_val = np.array(X_val),np.array(Y_val)

    # Adding Layers to the model
    model = Sequential()
    model.add(LSTM(X_train.shape[2],input_shape = (X_train.shape[1],X_train.shape[2]),return_sequences = True,
                   activation = 'relu'))
    for i in range(len(hl)-1):
        model.add(LSTM(hl[i],activation = 'relu',return_sequences = True))
    model.add(LSTM(hl[-1],activation = 'relu'))
    model.add(Dense(1))
    model.compile(optimizer = optimizers.Adam(lr = lr), loss = 'mean_squared_error')
    #print(model.summary())

    # Training the data
    history = model.fit(X_train,Y_train,epochs = epochs,batch_size = batch,validation_data = (X_val, Y_val),verbose = 0,
                        shuffle = False, callbacks=callbacks_list)
    model.reset_states()
    return model, history.history['loss'], history.history['val_loss']

# Evaluating the model
def evaluate_model(model,test,timesteps):
    X_test = []
    Y_test = []

    # Loop for testing data
    for i in range(timesteps,test.shape[0]):
        X_test.append(test[i-timesteps:i])
        Y_test.append(test[i][0])
    X_test,Y_test = np.array(X_test),np.array(Y_test)
    #print(X_test.shape,Y_test.shape)

    # Prediction Time !!!!
    Y_hat = model.predict(X_test)
    mse = mean_squared_error(Y_test,Y_hat)
    rmse = sqrt(mse)
    r = r2_score(Y_test,Y_hat)
    return mse, rmse, r, Y_test, Y_hat

# Plotting the predictions
def plot_data(Y_test,Y_hat):
    plt.plot(Y_test,c = 'r')
    plt.plot(Y_hat,c = 'y')
    plt.xlabel('Day')
    plt.ylabel('Price')
    plt.title('Stock Prediction Graph using Multivariate-LSTM model')
    plt.legend(['Actual','Predicted'],loc = 'lower right')
    plt.show()

# Plotting the training errors
def plot_error(train_loss,val_loss):
    plt.plot(train_loss,c = 'r')
    plt.plot(val_loss,c = 'b')
    plt.ylabel('Loss')
    plt.legend(['train','val'],loc = 'upper right')
    plt.show()




In [6]:
# Extracting the series
series = df[['Close','High','Volume']] # Picking the series with high correlation
print(series.shape)
print(series.tail())

(5320, 3)
                 Close        High    Volume
Date                                        
2023-11-23  923.000000  925.349976   5534614
2023-11-24  929.400024  930.400024   7452002
2023-11-28  925.500000  931.049988  14231987
2023-11-29  939.599976  941.099976  11680078
2023-11-30  934.950012  940.849976  23950292


In [7]:
# Train Val Test Split
train_start = dt.date(2000,1,1)
train_end = dt.date(2021,12,31)
train_data = series.loc[train_start:train_end]

val_start = dt.date(2022,1,1)
val_end = dt.date(2022,12,31)
val_data = series.loc[val_start:val_end]

test_start = dt.date(2023,1,1)
test_end = dt.date(2023,11,30)
test_data = series.loc[test_start:test_end]

print(train_data.shape,val_data.shape,test_data.shape)

(4847, 3) (248, 3) (225, 3)


In [8]:
# Normalisation
sc = MinMaxScaler()
train = sc.fit_transform(train_data)
val = sc.transform(val_data)
test = sc.transform(test_data)
print(train.shape,val.shape,test.shape)

(4847, 3) (248, 3) (225, 3)


In [9]:
timesteps = 50
hl = [40,35]
lr = 1e-3
batch_size = 64
num_epochs = 250

In [10]:
model,train_error,val_error = fit_model(train,val,timesteps,hl,lr,batch_size,num_epochs)
plot_error(train_error,val_error)

c:\Users\Vibhor\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\optimizers\legacy\adam.py:117: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


KeyboardInterrupt: 